[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nekrut/bda/blob/colab/lectures/lecture13.ipynb)

# Lecture 13: Sequence Alignment

> This lecture draws on material from:
> - [Sequence Alignment lecture notes](https://hackmd.io/@nekrut/BkVEKymzn) by Anton Nekrutenko
> - [Dynamic Programming and Edit Distance](http://www.cs.jhu.edu/~langmea/resources/lecture_notes/11_dp_and_edit_dist_v2.pdf) by Ben Langmead (JHU)
> - [Global Alignment](http://www.cs.jhu.edu/~langmea/resources/lecture_notes/12_global_alignment_v2.pdf) by Ben Langmead (JHU)
> - [Local Alignment](http://www.cs.jhu.edu/~langmea/resources/lecture_notes/13_local_alignment_v2.pdf) by Ben Langmead (JHU)
> - [DP in Less Time and Space](http://www.cs.jhu.edu/~langmea/resources/lecture_notes/14_dp_in_less_time_space_v2.pdf) by Ben Langmead (JHU)
> - [Computational Genomics notebooks](https://github.com/BenLangmead/comp-genomics-class) by Ben Langmead

Sequence alignment is one of the most fundamental operations in bioinformatics. It allows us to compare DNA, RNA, or protein sequences to identify regions of similarity that may reflect functional, structural, or evolutionary relationships. In this lecture we will build up from the concept of **edit distance** to full **global** and **local** alignment algorithms.

## Why align sequences?

Almost every task in genomics involves comparing sequences:

- **Read mapping** — aligning short sequencing reads to a reference genome
- **Variant calling** — finding differences between a sample and a reference
- **Homology search** — finding genes in other species that are related to a query (e.g., BLAST)
- **Genome assembly** — finding overlaps between reads
- **Phylogenetics** — building evolutionary trees from multiple sequence alignments

All of these rely on the ability to *optimally* align two (or more) sequences.

### The biology behind substitutions and indels

![Transitions vs transversions in DNA](../images/lecture13_transitions.png)

Not all mutations are equally likely:

- **Transitions** (purine↔purine: A↔G, or pyrimidine↔pyrimidine: C↔T) are chemically more similar changes and occur roughly **twice as often** as **transversions** (purine↔pyrimidine) in the human genome. In coding regions the ratio is even higher.
- **Insertions and deletions (indels)** are less common than substitutions — a single-nucleotide indel occurs approximately every **1,000 bases**, and a two-nucleotide deletion about every **3,000 bases** (Montgomery et al. 2013).

These biological realities directly inform how we design scoring and penalty functions for alignment algorithms.

# Part 1: Edit Distance

## What is edit distance?

![The three edit operations: substitution, insertion, and deletion](../images/lecture13_edit_ops.png)

The **edit distance** (also called **Levenshtein distance**) between two strings is the minimum number of single-character operations required to transform one string into the other. The allowed operations are:

| Operation | Example | Cost |
|-----------|---------|------|
| **Substitution** (replace) | `A` → `G` | 1 |
| **Insertion** (gap in first sequence) | `-` → `A` | 1 |
| **Deletion** (gap in second sequence) | `A` → `-` | 1 |

For example, to transform `ACGT` into `AGT`:
```
ACGT
A-GT    (delete C → cost 1)
```
The edit distance is **1**.

## Edit distance vs. Hamming distance

You may have encountered **Hamming distance** before — it counts the number of positions where two *equal-length* strings differ (substitutions only). Edit distance is more general because it allows insertions and deletions, so the strings can have different lengths.

**Key property:** For equal-length strings, $\text{editDistance}(x, y) \le \text{hammingDistance}(x, y)$. This is because edit distance can always "do at least as well" by considering insertions and deletions alongside substitutions.

A useful lower bound: $\text{editDistance}(x, y) \ge \big| |x| - |y| \big|$ — at minimum you need enough indels to account for any length difference.

In [1]:
def hamming_distance(x, y):
    """Return Hamming distance between two equal-length strings."""
    assert len(x) == len(y), "Strings must be the same length"
    return sum(c1 != c2 for c1, c2 in zip(x, y))

print(hamming_distance('ACGT', 'ACTT'))  # 1 substitution (G→T)

1


But what if the strings have different lengths?

```python
hamming_distance('ACGT', 'AGT')  # ERROR — different lengths!
```

This is where edit distance comes in.

## A naive recursive solution

We can compute edit distance recursively. The idea: compare the last characters of both strings, and consider all three operations:

$$
\text{edDist}(x, y) = \min \begin{cases}
\text{edDist}(x[:-1], y[:-1]) + \delta(x[-1], y[-1]) & \text{(substitution/match)} \\
\text{edDist}(x[:-1], y) + 1 & \text{(deletion from } x\text{)} \\
\text{edDist}(x, y[:-1]) + 1 & \text{(insertion into } x\text{)}
\end{cases}
$$

where $\delta(a, b) = 0$ if $a = b$, and $1$ otherwise.

In [2]:
def ed_recursive(x, y):
    """Compute edit distance between x and y recursively (naive)."""
    if len(x) == 0: return len(y)
    if len(y) == 0: return len(x)
    delt = 1 if x[-1] != y[-1] else 0
    return min(
        ed_recursive(x[:-1], y[:-1]) + delt,  # substitute / match
        ed_recursive(x[:-1], y) + 1,           # delete from x
        ed_recursive(x, y[:-1]) + 1            # insert into x
    )

print(ed_recursive('ACGT', 'AGT'))   # 1
print(ed_recursive('shake', 'spear'))  # 4

1
4


### Edit transcripts

An **edit transcript** records the step-by-step operations an "optimal editor" uses to transform one string into another, working left-to-right:

| Symbol | Meaning |
|--------|---------|
| **M** | Match (characters are equal) |
| **R** | Replace (substitution) |
| **D** | Delete from $x$ |
| **I** | Insert into $x$ |

For example, aligning `GCGTATGC` to `GCTATGCC`:
```
x: G C G T A T G - C
   | |   | | | |   |
y: G C - T A T G C C
```
The transcript is `MMDMMMMIM` and the edit distance is **2** (one D + one I).

### How the recursion works

The algorithm works **backwards from the last characters** of both strings and asks: *what is the cheapest way to deal with these two characters?*

There are exactly three choices at each step:

1. **Match / Substitute** — Align `x[-1]` with `y[-1]`. If they’re the same, it’s free (match). If they differ, it costs 1 (substitution). Either way, both last characters are handled, so recurse on `x[:-1]` vs `y[:-1]`.

2. **Delete from x** — The last character of `x` has no partner in `y`. It costs 1 to delete it. Now solve `x[:-1]` vs `y` (all of `y` still needs to be matched).

3. **Insert into x** — You need `y[-1]` but `x` doesn’t provide it, so insert it (cost 1). That handles `y[-1]`, so recurse on `x` vs `y[:-1]` (`x` is unchanged since the inserted character “consumed” `y[-1]`).

You try all three and take the **minimum**.

The recursion bottoms out when either string is empty:
- `x` empty → insert all remaining characters of `y` (cost = `len(y)`)
- `y` empty → delete all remaining characters of `x` (cost = `len(x)`)

### Worked example: `ed("ACT", "AT")`

We want the minimum number of edits to turn `ACT` into `AT`. The answer is **1** (delete `C`), but let’s see how the recursion finds it.

**Step 1** — `ed("ACT", "AT")`: compare last chars `T` vs `T` — they match!

| Branch | Operation | Recurse on | Cost so far |
|--------|-----------|------------|-------------|
| match `T`=`T` | free, handle both | `ed("AC", "A")` | +0 |
| delete `T` from x | remove `T` from end of x | `ed("AC", "AT")` | +1 |
| insert `T` into x | append `T` to match y’s last | `ed("ACT", "A")` | +1 |

**Step 2** — follow the match branch into `ed("AC", "A")`: compare `C` vs `A` — they differ.

| Branch | Operation | Recurse on | Cost so far |
|--------|-----------|------------|-------------|
| substitute `C`→`A` (+1) | replace last char | `ed("A", "")` | +1 |
| delete `C` from x (+1) | drop the `C` | `ed("A", "A")` | +1 |
| insert `A` into x (+1) | append `A` to match y’s last | `ed("AC", "")` | +1 |

**Step 3** — follow the delete branch into `ed("A", "A")`: compare `A` vs `A` — match!

| Branch | Operation | Recurse on | Cost so far |
|--------|-----------|------------|-------------|
| match `A`=`A` | free | `ed("", "")` | +0 |
| delete `A` from x (+1) | | `ed("", "A")` | +1 |
| insert `A` into x (+1) | | `ed("A", "")` | +1 |

**Step 4** — `ed("", "")`: both strings empty → cost **0**. Done!

Now unwind the results:

```
ed("", "")    = 0                         (both empty)
ed("A", "A")  = min(0+0, 0+1+1, 1+1) = 0  (match ‘A’=‘A’)
ed("AC", "A") = min(1+1, 0+1, 2+1)   = 1  (delete ‘C’ from x)
ed("ACT","AT")= min(1+0, ?+1, ?+1)   = 1  (match ‘T’=‘T’)
```

The optimal path was: **match `T`** → **delete `C`** → **match `A`** → done. This corresponds to the alignment:
```
x: A C T
y: A - T
```

But notice — the algorithm doesn’t just explore this one path. It tries **all three branches at every step**, which means `ed("A", "A")` and other subproblems get recomputed multiple times. This is what makes it exponentially slow.

### Why is this so slow?

This recursive approach is correct, but **extremely** slow. Here's why:

Every call to `ed_recursive` makes **three** more calls (one for each operation — match/substitute, delete, insert). Each of those makes three more calls, and so on. This creates a tree of calls that branches by a factor of 3 at every level.

Notice that when the two characters **don't match**, all three operations cost exactly +1: substitution costs 1 (it's a mismatch), deletion costs 1, and insertion costs 1. Since the local penalty is the same across all three branches, the only thing that determines which branch "wins" is what happens in the **remaining subproblem** — substitution shrinks both strings by one character, deletion shrinks only `x`, and insertion shrinks only `y`. The algorithm has no way to know which subproblem leads to the best answer without actually computing all three. When the characters **do** match, substitution gets a discount (cost +0 instead of +1), giving it an advantage — but the algorithm still evaluates all three branches to be sure.

For two strings of length $n$, the recursion can go about $2n$ levels deep before hitting a base case. That means the total number of calls grows like $3^{2n}$ — roughly **exponential**. Even for short strings like `"Shakespeare"` vs `"shake spear"` (11 characters each), the function takes several seconds.

The real problem is **redundant work**. Many of the subproblems are identical. For example, when computing `ed("ACGT", "AGT")`, the subproblem `ed("A", "A")` gets solved multiple times through different paths in the recursion tree. With longer strings this redundancy is massive — when comparing two 9-character sequences, the subproblem `ed("AC", "AC")` alone is recomputed **48,639 times**.

To see this concretely, here is what happens with just `ed("AC", "AG")`:

```
ed("AC","AG")                          ← top-level call
├── ed("A","A")     + 1  (sub: C≠G)   ← branch 1: match/substitute
│   ├── ed("","")   + 0               ← match A=A → returns 0
│   ├── ed("","A")  + 1               ← delete A  → returns 1+1=2
│   └── ed("A","")  + 1               ← insert A  → returns 1+1=2
│   → min(0, 2, 2) = 0, then + 1 = 1
├── ed("A","AG")    + 1  (delete C)    ← branch 2: delete from x
│   ├── ed("","A")  + 0               ← match A=A... but wait,
│   │                                    this needs ed("","A") AGAIN
│   ├── ed("","AG") + 1               ← delete A
│   └── ed("A","A") + 1               ← insert G  ← DUPLICATE!
│   → min(1, 2, 1) = 1, then + 1 = 2
├── ed("AC","A")    + 1  (insert G)    ← branch 3: insert into x
│   ├── ed("A","")  + 1               ← sub C≠A
│   ├── ed("A","A") + 1               ← delete C  ← DUPLICATE!
│   └── ed("AC","") + 1               ← insert A
│   → min(2, 1, 3) = 1, then + 1 = 2
→ min(1, 2, 2) = 1
```

Even in this tiny example, `ed("A","A")` is computed **3 times** and `ed("","A")` is computed **twice**. With longer strings, the same subproblem can be recomputed thousands or millions of times. This is the hallmark of a problem that can be dramatically sped up with **dynamic programming**.

### Memoization: a stepping stone

Before jumping to the full DP table, we can fix the naive recursion by **caching** results — this is called **memoization** (top-down dynamic programming):

In [3]:
import time

def ed_memoized(x, y, memo=None):
    """Edit distance with memoization (top-down DP)."""
    if memo is None:
        memo = {}
    if len(x) == 0: return len(y)
    if len(y) == 0: return len(x)
    if (len(x), len(y)) in memo:
        return memo[(len(x), len(y))]
    delt = 1 if x[-1] != y[-1] else 0
    ans = min(
        ed_memoized(x[:-1], y[:-1], memo) + delt,
        ed_memoized(x[:-1], y, memo) + 1,
        ed_memoized(x, y[:-1], memo) + 1
    )
    memo[(len(x), len(y))] = ans
    return ans

# Compare performance
t0 = time.time()
d1 = ed_recursive('Shakespeare', 'shake spear')
t1 = time.time()

t2 = time.time()
d2 = ed_memoized('Shakespeare', 'shake spear')
t3 = time.time()

print(f"Naive recursive: distance={d1}, time={t1-t0:.3f}s")
print(f"Memoized:        distance={d2}, time={t3-t2:.6f}s")

Naive recursive: distance=3, time=4.096s
Memoized:        distance=3, time=0.000090s


## Dynamic programming to the rescue

![How the DP matrix is filled — each cell is computed from three neighbors, and the traceback (red arrows) recovers the optimal alignment](../images/lecture13_dp_matrix.png)

**Dynamic programming (DP)** avoids redundant computation by building a table of solutions to subproblems. We create an $(m+1) \times (n+1)$ matrix $D$ where $D[i][j]$ holds the edit distance between the first $i$ characters of $x$ and the first $j$ characters of $y$.

### Initialization

- $D[i][0] = i$ — transforming $i$ characters into an empty string requires $i$ deletions
- $D[0][j] = j$ — transforming an empty string into $j$ characters requires $j$ insertions

### Recurrence

$$D[i][j] = \min \begin{cases} D[i-1][j-1] + \delta(x[i-1], y[j-1]) \\ D[i-1][j] + 1 \\ D[i][j-1] + 1 \end{cases}$$

The answer is in $D[m][n]$.

In [4]:
import numpy as np

def edit_distance(x, y):
    """Compute edit distance between strings x and y using dynamic programming."""
    D = np.zeros((len(x) + 1, len(y) + 1), dtype=int)
    
    # Base cases
    D[0, 1:] = range(1, len(y) + 1)
    D[1:, 0] = range(1, len(x) + 1)
    
    # Fill the matrix
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            delt = 1 if x[i-1] != y[j-1] else 0
            D[i, j] = min(
                D[i-1, j-1] + delt,  # diagonal: substitution or match
                D[i-1, j] + 1,       # vertical: deletion
                D[i, j-1] + 1        # horizontal: insertion
            )
    return D

x, y = 'GCGTATGC', 'TATTGGCTATGCG'
D = edit_distance(x, y)
print(f"Edit distance between '{x}' and '{y}': {D[len(x), len(y)]}")

Edit distance between 'GCGTATGC' and 'TATTGGCTATGCG': 7


### Visualizing the DP matrix

Let's write a helper to display the matrix nicely.

In [5]:
import pandas as pd

def display_matrix(D, x, y):
    """Display a DP matrix as a labeled DataFrame."""
    cols = ['-'] + list(y)
    rows = ['-'] + list(x)
    return pd.DataFrame(D, index=rows, columns=cols)

x, y = 'ACGT', 'AGT'
D = edit_distance(x, y)
display_matrix(D, x, y)

,-,A,G,T
-,0,1,2,3
A,1,0,1,2
C,2,1,1,2
G,3,2,1,2
T,4,3,2,1


Reading the matrix:
- The top-left corner (0,0) is the base case: two empty strings → distance 0
- The bottom-right corner gives the final answer
- Each cell was computed from the three neighbors: **diagonal** (↖), **above** (↑), and **left** (←)

# Part 2: Global Alignment (Needleman-Wunsch)

Edit distance treats all operations equally (cost = 1). But in biology, not all changes are equally likely:

- **Transitions** (purine ↔ purine: A↔G, or pyrimidine ↔ pyrimidine: C↔T) are more common than **transversions** (purine ↔ pyrimidine)
- **Gaps** (insertions/deletions) may be penalized more heavily than substitutions

**Global alignment** aligns two sequences end-to-end using a **penalty/scoring matrix** that reflects these biological realities.

The **Needleman-Wunsch** algorithm (1970) solves this using dynamic programming — it is essentially edit distance with a more sophisticated cost function.

## Defining a penalty function

We define a cost function where:
- **Match** = 0 (no penalty)
- **Transition** = 2
- **Transversion** = 4
- **Gap** = 8

This is a *penalty-based* scheme (lower is better), unlike the *score-based* scheme we will use for local alignment later.

### Where do these numbers come from?

Penalty functions are informed by three complementary criteria:

1. **Mutational frequency** — penalties inversely related to how often a mutation type occurs in nature (transitions are ~4× more common than expected by chance, so they are penalized less)
2. **Biochemical interchangeability** — residues that serve similar roles (e.g., both hydrophobic) incur lower penalties when swapped
3. **Structural impact** — substitutions that preserve 3D fold and function cost less than those that disrupt them

For DNA, the 0/2/4/8 scheme reflects the empirical observation that substitutions occur ~1 in 1,000 bases while indels occur ~1 in 3,000 bases (so gaps cost ~3× more), and transitions are more common than transversions.

For **protein** sequences, the widely used **BLOSUM62** matrix encodes log-odds scores for all 20×20 amino acid substitutions, derived from observed substitution patterns in conserved protein blocks. Positive values mean the substitution is more common than expected (benign), while negative values indicate it is rare (disruptive).

In [6]:
def penalty(xc, yc):
    """Penalty function for global alignment.
    Match=0, transition=2, transversion=4, gap=8."""
    if xc == yc: 
        return 0                          # match
    if xc == '-' or yc == '-': 
        return 8                          # gap
    minc, maxc = min(xc, yc), max(xc, yc)
    if (minc == 'A' and maxc == 'G') or (minc == 'C' and maxc == 'T'):
        return 2                          # transition
    return 4                              # transversion

# Demonstrate
print(f"A vs A (match):       {penalty('A', 'A')}")
print(f"A vs G (transition):  {penalty('A', 'G')}")
print(f"A vs C (transversion):{penalty('A', 'C')}")
print(f"A vs - (gap):         {penalty('A', '-')}")

A vs A (match):       0
A vs G (transition):  2
A vs C (transversion):4
A vs - (gap):         8


## The Needleman-Wunsch algorithm

The algorithm is almost identical to edit distance DP, but uses our custom penalty function instead of a uniform cost of 1:

$$D[i][j] = \min \begin{cases} D[i-1][j-1] + \text{penalty}(x[i-1], y[j-1]) & \text{(match/mismatch)} \\ D[i-1][j] + \text{penalty}(x[i-1], \texttt{'-'}) & \text{(gap in } y\text{)} \\ D[i][j-1] + \text{penalty}(\texttt{'-'}, y[j-1]) & \text{(gap in } x\text{)} \end{cases}$$

The optimal **global** alignment score is in $D[m][n]$.

In [7]:
def global_alignment(x, y, s):
    """Needleman-Wunsch global alignment.
    
    Args:
        x, y: sequences to align
        s: penalty function s(char1, char2) -> cost
    
    Returns:
        D: the filled DP matrix
        score: the optimal alignment score (bottom-right cell)
    """
    D = np.zeros((len(x) + 1, len(y) + 1), dtype=int)
    
    # Initialize first row and column with gap penalties
    for j in range(1, len(y) + 1):
        D[0, j] = D[0, j-1] + s('-', y[j-1])
    for i in range(1, len(x) + 1):
        D[i, 0] = D[i-1, 0] + s(x[i-1], '-')
    
    # Fill the matrix
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            D[i, j] = min(
                D[i-1, j-1] + s(x[i-1], y[j-1]),  # diagonal
                D[i-1, j]   + s(x[i-1], '-'),      # vertical (gap in y)
                D[i, j-1]   + s('-', y[j-1])       # horizontal (gap in x)
            )
    
    return D, D[len(x), len(y)]

In [8]:
x = 'TACGTCAGC'
y = 'TATGTCATGC'

D, score = global_alignment(x, y, penalty)
print(f"Global alignment penalty for '{x}' vs '{y}': {score}")
print()
display_matrix(D, x, y)

Global alignment penalty for 'TACGTCAGC' vs 'TATGTCATGC': 10



,-,T,A,T,G,T,C,A,T,G,C
-,0,8,16,24,32,40,48,56,64,72,80
T,8,0,8,16,24,32,40,48,56,64,72
A,16,8,0,8,16,24,32,40,48,56,64
C,24,16,8,2,10,18,24,32,40,48,56
G,32,24,16,10,2,10,18,26,34,40,48
T,40,32,24,16,10,2,10,18,26,34,42
C,48,40,32,24,18,10,2,10,18,26,34
A,56,48,40,32,26,18,10,2,10,18,26
G,64,56,48,40,32,26,18,10,6,10,18
C,72,64,56,48,40,34,26,18,12,10,10


The penalty of 10 represents the best possible end-to-end alignment of these two sequences. The low penalty is driven by the many matches.

## Traceback: recovering the alignment

The DP matrix tells us the *score*, but not the actual alignment. To recover the alignment, we **trace back** from $D[m][n]$ to $D[0][0]$, following the path that produced each cell's value.

In [9]:
def global_alignment_traceback(D, x, y, s):
    """Traceback through a global alignment DP matrix to recover the alignment."""
    i, j = len(x), len(y)
    aligned_x, aligned_y, midline = [], [], []
    
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i-1, j-1] + s(x[i-1], y[j-1]):
            # Diagonal: match or substitution
            aligned_x.append(x[i-1])
            aligned_y.append(y[j-1])
            midline.append('|' if x[i-1] == y[j-1] else ' ')
            i -= 1; j -= 1
        elif i > 0 and D[i, j] == D[i-1, j] + s(x[i-1], '-'):
            # Vertical: gap in y
            aligned_x.append(x[i-1])
            aligned_y.append('-')
            midline.append(' ')
            i -= 1
        else:
            # Horizontal: gap in x
            aligned_x.append('-')
            aligned_y.append(y[j-1])
            midline.append(' ')
            j -= 1
    
    # Reverse (we traced back from end to start)
    aligned_x = ''.join(reversed(aligned_x))
    aligned_y = ''.join(reversed(aligned_y))
    midline   = ''.join(reversed(midline))
    
    return aligned_x, midline, aligned_y

ax, mid, ay = global_alignment_traceback(D, x, y, penalty)
print(f"  x: {ax}")
print(f"     {mid}")
print(f"  y: {ay}")

  x: TACGTCA-GC
     || |||| ||
  y: TATGTCATGC


The `|` symbols mark matching positions. Gaps (`-`) represent insertions or deletions.

## Try it: a longer example

In [10]:
seq1 = 'ATAGACGACATACAGACAGCATACAGACAGCATACAGA'
seq2 = 'TTTAGCATGCGCATATCAGCAATACAGCAGATACG'

D, score = global_alignment(seq1, seq2, penalty)
ax, mid, ay = global_alignment_traceback(D, seq1, seq2, penalty)

print(f"Penalty: {score}\n")
print(f"  x: {ax}")
print(f"     {mid}")
print(f"  y: {ay}")

Penalty: 74

  x: ATAGACGACATACAGACAGCATACAGACAGCATACAGA
      |  | | ||| |  | | ||   | |||||| | |  
  y: -TTTA-G-CATGCGCATATCAGCAATACAGCAGATACG


# Part 3: Local Alignment (Smith-Waterman)

Global alignment forces the entire length of both sequences to participate in the alignment. But what if we have two long sequences that share only a short region of high similarity?

**Local alignment** finds the *best matching subsequences* within two longer sequences. The **Smith-Waterman** algorithm (1981) is the standard method.

### Key differences from global alignment

| Feature | Global (Needleman-Wunsch) | Local (Smith-Waterman) |
|---------|--------------------------|------------------------|
| Aligns | Entire sequences end-to-end | Best matching subregions |
| Scoring | Penalties (lower is better) | Rewards (higher is better) |
| Matrix init | First row/column = gap costs | First row/column = 0 |
| Cell minimum | Can be negative | Clamped to 0 (reset) |
| Answer location | Bottom-right cell | Maximum value anywhere in matrix |
| Traceback | From bottom-right to top-left | From max cell until reaching 0 |

## Scoring function

For local alignment, we use a **reward-based** scoring scheme (higher is better):
- **Match** = +2 (reward)
- **Mismatch** = -4 (penalty)
- **Gap** = -6 (penalty)

The positive match reward is critical — it allows good regions to accumulate a high score.

### Why must matches be positive and edits negative?

The zero floor in Smith-Waterman only works if:
- Matches push the score **above** zero (positive bonus)
- Mismatches and gaps push it **below** zero (negative penalty)

If matches were not positive, all cells would equal 0 — the floor would always dominate. If edits were not negative, the floor would never be reached and the algorithm would degenerate to global alignment. The zeros in the matrix act as a "background level" that allows peaks of similarity to rise above.

> **Note on duality:** Penalty-based (minimize) and reward-based (maximize) scoring produce *identical tracebacks* for global alignment — they are equivalent formulations related by a sign flip plus constant. However, local alignment *requires* the reward-based formulation so the zero-floor reset is meaningful.

In [11]:
def local_score(xc, yc):
    """Scoring function for local alignment.
    Match=+2, mismatch=-4, gap=-6."""
    if xc == yc:
        return 2    # match reward
    if xc == '-' or yc == '-':
        return -6   # gap penalty
    return -4       # mismatch penalty

## The Smith-Waterman algorithm

The recurrence is:

$$V[i][j] = \max \begin{cases} V[i-1][j-1] + s(x[i-1], y[j-1]) & \text{(diagonal)} \\ V[i-1][j] + s(x[i-1], \texttt{'-'}) & \text{(vertical)} \\ V[i][j-1] + s(\texttt{'-'}, y[j-1]) & \text{(horizontal)} \\ 0 & \text{(reset — start a new alignment)} \end{cases}$$

The **0** option is the key innovation: it allows the algorithm to abandon a poor alignment and start fresh, thereby finding local regions of high similarity.

In [12]:
def smith_waterman(x, y, s):
    """Smith-Waterman local alignment.
    
    Args:
        x, y: sequences to align
        s: scoring function s(char1, char2) -> score
    
    Returns:
        V: the filled DP matrix
        max_score: the best local alignment score
    """
    V = np.zeros((len(x) + 1, len(y) + 1), dtype=int)
    
    # Note: first row and column stay 0 (no gap initialization)
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            V[i, j] = max(
                V[i-1, j-1] + s(x[i-1], y[j-1]),  # diagonal
                V[i-1, j]   + s(x[i-1], '-'),      # vertical
                V[i, j-1]   + s('-', y[j-1]),       # horizontal
                0                                   # reset
            )
    
    max_score = int(V.max())
    return V, max_score

In [13]:
x = 'GGTATGCTGGCGCTA'
y = 'TATATGCGGCGTTT'

V, max_score = smith_waterman(x, y, local_score)
print(f"Best local alignment score: {max_score}")
print()
display_matrix(V, x, y)

Best local alignment score: 12



,-,T,A,T,A,T,G,C,G,G,C,G,T,T,T
-,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
G,0,0,0,0,0,0,2,0,2,2,0,2,0,0,0
G,0,0,0,0,0,0,2,0,2,4,0,2,0,0,0
T,0,2,0,2,0,2,0,0,0,0,0,0,4,2,2
A,0,0,4,0,4,0,0,0,0,0,0,0,0,0,0
T,0,2,0,6,0,6,0,0,0,0,0,0,2,2,2
G,0,0,0,0,2,0,8,2,2,2,0,2,0,0,0
C,0,0,0,0,0,0,2,10,4,0,4,0,0,0,0
T,0,2,0,2,0,2,0,4,6,0,0,0,2,2,2
G,0,0,0,0,0,0,4,0,6,8,2,2,0,0,0


Notice the zeros scattered throughout the matrix — these are positions where the alignment "restarted." The maximum value in the matrix indicates the best local alignment.

## Traceback for local alignment

We start at the cell with the **maximum score** and trace back until we hit a cell with value **0**.

In [14]:
def sw_traceback(V, x, y, s):
    """Traceback through a Smith-Waterman matrix to recover the local alignment."""
    # Find the position of the maximum score
    i, j = np.unravel_index(np.argmax(V), V.shape)
    
    aligned_x, aligned_y, midline = [], [], []
    
    # Trace back until we hit 0
    while i > 0 and j > 0 and V[i, j] != 0:
        diag = V[i-1, j-1] + s(x[i-1], y[j-1]) if (i > 0 and j > 0) else -1
        vert = V[i-1, j]   + s(x[i-1], '-')     if i > 0 else -1
        horz = V[i, j-1]   + s('-', y[j-1])     if j > 0 else -1
        
        if diag >= vert and diag >= horz:
            match = x[i-1] == y[j-1]
            midline.append('|' if match else ' ')
            aligned_x.append(x[i-1])
            aligned_y.append(y[j-1])
            i -= 1; j -= 1
        elif vert >= horz:
            aligned_x.append(x[i-1])
            aligned_y.append('-')
            midline.append(' ')
            i -= 1
        else:
            aligned_x.append('-')
            aligned_y.append(y[j-1])
            midline.append(' ')
            j -= 1
    
    aligned_x = ''.join(reversed(aligned_x))
    aligned_y = ''.join(reversed(aligned_y))
    midline   = ''.join(reversed(midline))
    
    return aligned_x, midline, aligned_y

In [15]:
ax, mid, ay = sw_traceback(V, x, y, local_score)

print(f"Best local alignment (score = {max_score}):\n")
print(f"  x: {ax}")
print(f"     {mid}")
print(f"  y: {ay}")

Best local alignment (score = 12):

  x: TATGCTGGCG
     ||||| ||||
  y: TATGC-GGCG


The local alignment picks out just the high-similarity region, ignoring the poorly-matching flanks.

# Part 4: Global vs. Local — A Side-by-Side Comparison

![Global alignment uses the full length of both sequences; local alignment finds the best matching subregion](../images/lecture13_global_vs_local.png)

Let's see how global and local alignment handle the same pair of sequences.

In [ ]:
seq_x = 'AACCTTGGCTAACTGG'
seq_y = 'ACACTGTGA'

# --- Global alignment ---
D_g, score_g = global_alignment(seq_x, seq_y, penalty)
ax_g, mid_g, ay_g = global_alignment_traceback(D_g, seq_x, seq_y, penalty)

print("=== Global Alignment (Needleman-Wunsch) ===")
print(f"Penalty: {score_g}\n")
print(f"  x: {ax_g}")
print(f"     {mid_g}")
print(f"  y: {ay_g}")

print()

# --- Local alignment ---
V_l, score_l = smith_waterman(seq_x, seq_y, local_score)
ax_l, mid_l, ay_l = sw_traceback(V_l, seq_x, seq_y, local_score)

print("=== Local Alignment (Smith-Waterman) ===")
print(f"Score: {score_l}\n")
print(f"  x: {ax_l}")
print(f"     {mid_l}")
print(f"  y: {ay_l}")

Notice the difference:
- **Global alignment** forces both entire sequences to align, introducing gaps to accommodate the length difference
- **Local alignment** finds the best-matching subregion and ignores the rest

## Complexity

Both Needleman-Wunsch and Smith-Waterman have:
- **Time complexity**: $O(m \times n)$ — we fill every cell in the matrix
- **Space complexity**: $O(m \times n)$ — we store the entire matrix

For two sequences of length 1,000, that's a million operations — very manageable. But for genome-scale comparisons (millions of bases), this becomes prohibitively expensive.

Note that although there are $O(m^2 n^2)$ possible substring pairs to consider, Smith-Waterman solves the local alignment problem in just $O(mn)$ — a remarkable efficiency gain from the DP formulation.

## Heuristic alternatives

Because exact alignment is expensive, practical tools use **heuristic** approaches:

- **BLAST** — finds short exact matches (seeds), then extends them with DP only in promising regions
- **Minimap2** — uses minimizer-based seeding for fast long-read alignment
- **BWA-MEM** — uses a suffix array and seed-and-extend for short-read mapping
- **LASTZ** — whole genome alignment using seeded DP
- **MUMmer** — uses suffix trees for rapid whole-genome comparison

These tools trade some optimality for enormous speed gains, making it possible to align millions of reads to a genome in minutes.

## Glocal alignment: chaining local alignments

For whole-genome comparison, neither pure global nor pure local alignment is ideal. **Glocal alignment** chains multiple local alignments together to cover the full extent of both genomes while allowing rearrangements, inversions, and unaligned regions. This is the approach used by tools like MUMmer and the chaining step in MultiZ.

## Space optimization

The $O(mn)$ space requirement can be reduced:

**Two-row trick:** To compute the score alone (without traceback), we only need the current and previous rows of the DP matrix. This reduces space to $O(\min(m,n))$.

**Hirschberg's algorithm:** Combines the two-row trick with divide-and-conquer to recover the *full alignment* in $O(\min(m,n))$ space while maintaining $O(mn)$ time. The key insight is to run the forward DP on the first half of one sequence, the reverse DP on the second half, and combine them to find the midpoint of the optimal alignment.

## Banded alignment

When two sequences are known to be similar (edit distance ≤ $k$), we only need to fill a diagonal band of width $2k+1$ in the DP matrix. Cells outside this band cannot be part of the optimal alignment. This reduces both time and space to $O(kn)$ instead of $O(mn)$ — a significant speedup when $k \ll m$.

## Gap penalty models

In our implementations, each gap position costs the same ("linear" gap penalty). Several gap penalty models exist:

| Model | Formula | Notes |
|-------|---------|-------|
| **Constant** | $g$ | Fixed cost regardless of gap length |
| **Linear** | $k \cdot g$ | Cost proportional to gap length $k$ |
| **Affine** | $g_\text{open} + k \cdot g_\text{extend}$ | Opening cost + extension cost per position |
| **Convex** | e.g., $\log(k)$ | Diminishing marginal cost; more biologically realistic |

In biology, a single long insertion/deletion event is more likely than multiple independent ones. **Affine gap penalties** model this with a large opening penalty and a smaller extension penalty, encouraging the algorithm to extend existing gaps rather than open new ones. Simple (linear, affine) models dominate in practice mostly because they can be computed efficiently, not because they are the most biologically realistic.

## Approximate matching

A closely related application is **approximate pattern matching**: given a short pattern $P$ and a longer text $T$, find all positions in $T$ where $P$ occurs with at most $k$ edits. The trick is to initialize the **first row** of the DP matrix to all zeros (instead of $0, 1, 2, \ldots$), which allows matches to begin at any position in the text without penalty.

In [ ]:
def approximate_match(pattern, text):
    """Find approximate occurrences of pattern in text using edit distance DP.
    
    Returns the DP matrix where the minimum value in the last row gives
    the best approximate match score.
    """
    D = np.zeros((len(pattern) + 1, len(text) + 1), dtype=int)
    
    # First row is all zeros (matches can start anywhere in text)
    # First column: transforming pattern[:i] to empty string costs i deletions
    D[1:, 0] = range(1, len(pattern) + 1)
    
    for i in range(1, len(pattern) + 1):
        for j in range(1, len(text) + 1):
            delt = 0 if pattern[i-1] == text[j-1] else 1
            D[i, j] = min(
                D[i-1, j-1] + delt,
                D[i-1, j] + 1,
                D[i, j-1] + 1
            )
    return D

text    = 'AACCCTATGTCATTGGA'
pattern = 'TACGTCAGC'

D = approximate_match(pattern, text)
best_score = D[len(pattern), :].min()
best_pos   = D[len(pattern), :].argmin()

print(f"Pattern: {pattern}")
print(f"Text:    {text}")
print(f"Best approximate match ends at position {best_pos} with {best_score} edits")
print()
display_matrix(D, pattern, text)

# Part 5b: Whole-Genome Alignment Pipelines

![From exact algorithms to whole-genome alignment pipelines](../images/lecture13_pipeline.png)

The algorithms above work on individual sequence pairs. Scaling alignment to entire genomes requires elaborate pipelines.

## The MultiZ pipeline

MultiZ builds **whole-genome multiple alignments** in several stages:

1. **Softmasking** — interspersed and tandem repeats are converted to lowercase. Alignment tools skip masked regions for seeding but can extend matches through them.
2. **Segmentation** — genomes are split into smaller segments for parallel processing.
3. **Pairwise alignment** — all reference segments are aligned against each query genome using **lastZ** (a pairwise local aligner).
4. **Chaining and netting** — co-linear alignment blocks are grouped into chains; nets select the best chain at each reference position.
5. **Multiple alignment** — all pairwise combinations are merged into a multiple alignment using MultiZ.
6. **Conservation analysis** — the resulting MAF (Multiple Alignment Format) is used to compute **phastCons** (conserved regions) and **phyloP** (per-base conservation scores).

## Progressive Cactus

An alternative approach uses a **guide tree** to decompose the multiple alignment into parallelizable subtasks:

1. At each internal node, a local alignment step (lastZ) and a base-level alignment refinement step (CAF/BAR) are performed.
2. Ancestral genomes are reconstructed at internal nodes by aligning ingroup species and using outgroups to resolve orthology.
3. Sub-alignments are stitched together into a genome-wide multiple alignment stored in **HAL format**.

Both pipelines produce alignments that underpin conservation tracks in genome browsers like the UCSC Genome Browser.

# Part 6: Putting It All Together — Interactive Examples

In [ ]:
def align_and_display(x, y, method='both'):
    """Run global and/or local alignment on sequences x and y, and display results."""
    if method in ('global', 'both'):
        D, score = global_alignment(x, y, penalty)
        ax, mid, ay = global_alignment_traceback(D, x, y, penalty)
        print("=== Global Alignment ===")
        print(f"Penalty: {score}")
        print(f"  x: {ax}")
        print(f"     {mid}")
        print(f"  y: {ay}")
        print()
    
    if method in ('local', 'both'):
        V, score = smith_waterman(x, y, local_score)
        ax, mid, ay = sw_traceback(V, x, y, local_score)
        print("=== Local Alignment ===")
        print(f"Score: {score}")
        print(f"  x: {ax}")
        print(f"     {mid}")
        print(f"  y: {ay}")

In [ ]:
# Example 1: Similar sequences
print("--- Example 1: Similar sequences ---")
align_and_display('ATCGATCGA', 'ATCAATCGA')

In [ ]:
# Example 2: Short match embedded in longer sequence
print("--- Example 2: Shared motif in different contexts ---")
align_and_display('XXXATCGATCGAXXX', 'YYYATCGATCGAYYY')

In [ ]:
# Example 3: Very different sequences
print("--- Example 3: Dissimilar sequences ---")
align_and_display('AAAAAGGGGG', 'CCCCCTTTTT')

# Exercises

1. **Modify the penalty function**: Create a penalty function where all mismatches cost the same (say, 3) and gaps cost 5. Re-run the global alignment on `TACGTCAGC` vs `TATGTCATGC`. How does the alignment change?

2. **Experiment with scoring**: Change the local alignment scoring to match=+1, mismatch=-1, gap=-2. Align `AAACCCGATTT` vs `CCCGAT`. Does the algorithm still find the shared region?

3. **Approximate matching**: Modify the edit distance function to allow approximate pattern matching — find all positions in a text $T$ where pattern $P$ occurs with at most $k$ edits. *Hint*: initialize the first row of the DP matrix to all zeros instead of $0, 1, 2, \ldots$

4. **Protein alignment**: The concepts extend to protein sequences. Write a scoring function for amino acids where identical amino acids score +3, similar amino acids (e.g., both hydrophobic) score +1, and dissimilar amino acids score -2. Use BLOSUM62 for inspiration.

## Summary

In this lecture we covered:

1. **Edit distance** — the minimum number of insertions, deletions, and substitutions to transform one string into another, and the concept of edit transcripts (M/R/D/I)
2. **Dynamic programming** — from naive recursion to memoization to the bottom-up DP table, avoiding exponential redundant computation
3. **Global alignment (Needleman-Wunsch)** — aligning sequences end-to-end with biologically motivated penalty functions informed by mutational frequency, biochemical similarity, and structural impact
4. **Local alignment (Smith-Waterman)** — finding the best-matching subregions between sequences using a reward-based scoring scheme with a zero floor
5. **Traceback** — recovering the actual alignment from the DP matrix
6. **Approximate matching** — adapting the DP framework to find approximate pattern occurrences in text
7. **Practical considerations** — complexity, space optimization (Hirschberg), banded DP, gap penalty models (linear, affine, convex), and heuristic tools (BLAST, minimap2, BWA-MEM)
8. **Whole-genome alignment** — pipelines like MultiZ and Progressive Cactus that scale pairwise alignment to entire genomes, producing conservation tracks (phastCons, phyloP)